In [ ]:
import importlib
import numpy as np
import gsd.hoomd

from md_Helpers import (
    paths, spatial, metadata, lattices, simulation, runs,
    classification, cavitation, cavitation_analysis,
    visualization, index,
)

for module in [
    paths, spatial, metadata, lattices, classification, runs,
    simulation, cavitation, cavitation_analysis, visualization, index,
]:
    importlib.reload(module)

print("All V3 modules imported successfully.")

In [ ]:
thermal = simulation.get_or_make_thermalized_state(
    n_fcc_cells=20,
    target_rho=0.800,
    kT=0.800,
    nsteps=100_000,
    overwrite=False,
)

initial = cavitation.get_or_create_cavitation_state(
    n_fcc_cells=20,
    target_rho=0.800,
    kT=0.800,
    source_nsteps=100_000,
    radius=1.7,
    overwrite=False,
)

assert initial["paths"]["state_path"].exists()
assert initial["paths"]["creation_metadata_path"].exists()
assert initial["creation_info"]["particles_removed"] > 0

visualization.plot_cavitation_xy_slice(initial, fraction=0.03)
initial["creation_info"]

In [ ]:
evolution = cavitation.get_or_create_cavitation(
    n_fcc_cells=30,
    target_rho=0.71,
    kT=0.80,
    source_nsteps=1_000_000,
    radius=4.0,
    evolve_nsteps=100_000,
)


with gsd.hoomd.open(evolution["paths"]["trajectory_path"], "r") as trajectory:
    trajectory_steps = [
        frame.configuration.step for frame in trajectory
    ]
    assert np.allclose(
        trajectory[0].particles.position,
        initial["frame"].particles.position,
    )

log = runs.read_hdf5_log(evolution["paths"]["log_path"])
log_steps = log["hoomd-data"]["Simulation"]["timestep"]

assert trajectory_steps[0] == log_steps[0]
print("Trajectory steps:", trajectory_steps)
print("Log steps:", log_steps)

In [ ]:
classification.classify_final_state(
    state_path=evolution["paths"]["final_state_path"],
    log_path=evolution["paths"]["log_path"],
)

measurements = cavitation_analysis.measure_cavitation_trajectory(
    evolution,
    save_csv_path=evolution["paths"]["folder"] / "cavitation_measurements.csv",
)

visualization.plot_cavitation_measurements(measurements)
display(measurements.head())

index_table = index.scan_v3_metadata()
print("Indexed rows:", len(index_table))
display(index_table.head())

# Requires pyarrow:
index.build_v3_index()

In [ ]:
from IPython.display import display
from md_Helpers import visualization

creation_info = evolution["initial_result"]["creation_info"]
center_z = creation_info.get(
    "bubble_center_z",
    creation_info.get("bubble_center", [0, 0, 0])[2],
)

display(
    visualization.animate_xy_slice_trajectory(
        evolution["paths"]["trajectory_path"],
        fraction=0.03,
        center_z=center_z,
        stride=1,
        max_frames=100,
        point_size=2,
        alpha=0.7,
        interval=120,
    )
)